In [1]:
import sys
import os
# Agrega el directorio padre (Source) al sys.path
sys.path.append(os.path.abspath('..'))
from raidcontrol.NumberCNN_OCR import DigitCNN

In [2]:
import os
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms



In [5]:
@dataclass
class Config:
    data_dir: str = r"E:\Documents\Facultad\Proyecto\Data\dataset\synthetic_number2"
    batch_size: int = 128
    epochs: int = 50
    lr: float = 1e-3
    num_workers: int = 0
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    out_path: str = "digit_cnn_64_v4.pt"


In [4]:
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


def main():
    cfg = Config()
    set_seed(cfg.seed)

    # Augmentations: keep them mild to avoid breaking digit shape at 28x28.
    train_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),

        # Small geometric jitter
        transforms.RandomAffine(
            degrees=5,
            translate=(0.10, 0.10),   # ~2-3 px
            scale=(0.95, 1.05),
            shear=5,
            fill=0
        ),
        transforms.RandomRotation(
            degrees=45,
            fill=0
        ),
        transforms.RandomPerspective(
            distortion_scale=0.4,
            p=0.5
        ),
        transforms.ElasticTransform(
            alpha=10.0,
        ),
        # Optional blur to simulate motion/out-of-focus
        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))],
            p=0.25
        ),

        transforms.ToTensor(),  # -> [0,1], shape 1x28x28

        # Light noise (works well for robustness)
        transforms.Lambda(lambda t: torch.clamp(t + 0.03 * torch.randn_like(t), 0.0, 1.0)),
    ])

    val_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
    ])

    train_dir = os.path.join(cfg.data_dir, "train")
    val_dir = os.path.join(cfg.data_dir, "val")

    train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
    val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=True)

    model = DigitCNN().to(cfg.device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)

    best_val_acc = 0.0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        train_loss = 0.0
        train_acc = 0.0

        for x, y in train_loader:
            x = x.to(cfg.device, non_blocking=True)
            y = y.to(cfg.device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc += accuracy(logits.detach(), y)

        train_loss /= max(1, len(train_loader))
        train_acc /= max(1, len(train_loader))

        model.eval()
        val_loss = 0.0
        val_acc = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(cfg.device, non_blocking=True)
                y = y.to(cfg.device, non_blocking=True)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item()
                val_acc += accuracy(logits, y)

        val_loss /= max(1, len(val_loader))
        val_acc /= max(1, len(val_loader))

        print(f"Epoch {epoch:02d}/{cfg.epochs} | "
              f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model_state": model.state_dict(),
                "class_to_idx": train_ds.class_to_idx,
            }, cfg.out_path)
            print(f"Saved best model to {cfg.out_path} (val acc {best_val_acc:.4f})")

    print(f"Done. Best val acc: {best_val_acc:.4f}")

In [6]:
main()

e:\Documents\Facultad\Proyecto\raidcontrol\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 01/50 | train loss 2.1436 acc 0.2624 | val loss 2.4709 acc 0.0990
Saved best model to digit_cnn_64_v4.pt (val acc 0.0990)
Epoch 02/50 | train loss 1.8094 acc 0.4756 | val loss 2.4839 acc 0.1367
Saved best model to digit_cnn_64_v4.pt (val acc 0.1367)
Epoch 03/50 | train loss 1.5040 acc 0.6108 | val loss 2.0087 acc 0.2314
Saved best model to digit_cnn_64_v4.pt (val acc 0.2314)
Epoch 04/50 | train loss 1.2533 acc 0.7088 | val loss 0.9590 acc 0.7965
Saved best model to digit_cnn_64_v4.pt (val acc 0.7965)
Epoch 05/50 | train loss 1.0362 acc 0.7773 | val loss 0.8779 acc 0.7262
Epoch 06/50 | train loss 0.8655 acc 0.8205 | val loss 0.6969 acc 0.7747
Epoch 07/50 | train loss 0.7405 acc 0.8406 | val loss 0.5525 acc 0.8793
Saved best model to digit_cnn_64_v4.pt (val acc 0.8793)
Epoch 08/50 | train loss 0.6261 acc 0.8582 | val loss 0.4840 acc 0.8669
Epoch 09/50 | train loss 0.5719 acc 0.8730 | val loss 0.6702 acc 0.7695
Epoch 10/50 | train loss 0.5178 acc 0.8804 | val loss 0.6013 acc 0.8002
